# Stuttering Detection: K-Nearest Neighbors (KNN) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Distance-Based Classification on WavLM Manifolds

---

## Step 1: Initialization

In [1]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import KNNModel

# Paths to our distributed dataset
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

/home/anshuman139/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Native randomized sampling of the dataset
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine

In [2]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Oversampling training data only
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Anti-Leakage Standard Selection (MANDATORY for KNN accuracy)
X_train_final = manager.preprocess(X_train_bal, method="standard", fit=True)
X_test_final = manager.preprocess(X_test, method="standard", fit=False)

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model Execution - K-Nearest Neighbors
We use **K=5** neighbors.

In [3]:
knn_model = KNNModel("KNN_K5", n_neighbors=5)
knn_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
knn_model.evaluate(X_test_final, y_test)

[Model: KNN_K5] Initialized.
[KNN_K5] Stores 4440 training vectors...

--- Evaluation on Unseen Test Set ---

--- Evaluation: KNN_K5 ---
Accuracy: 0.5992
Precision: 0.4355
Recall: 0.4075
F1: 0.4211

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      336             140            
True: Stutter(1)     157             108            


{'accuracy': 0.5991902834008097,
 'precision': 0.43548387096774194,
 'recall': 0.4075471698113208,
 'f1': 0.42105263157894735,
 'confusion_matrix': array([[336, 140],
        [157, 108]])}